# Velocity Fields in the Solar Atmosphere — Implementation
# 태양 대기의 속도장 — 구현

**Leighton, Noyes & Simon (1962)**

이 노트북에서는 논문의 핵심 개념과 분석 방법을 Python으로 재현합니다:
This notebook reproduces the key concepts and analysis methods from the paper in Python:

1. **Line-Shifter 시뮬레이션**: Doppler shift가 적색/청색 날개 밝기에 미치는 영향
2. **Supergranulation 속도장 생성 및 시선 투영**: 대류 셀 모델과 관측 투영 효과
3. **자기상관(A-C) 함수 분석**: 패턴 크기와 규칙성 정량화
4. **밝기–속도 상관 분석**: 고도에 따른 부호 반전 시뮬레이션
5. **5분 진동 시뮬레이션**: 감쇠 진동과 Doppler difference plate
6. **에너지 수송 추정**: 대류에 의한 기계적 에너지 수송
7. **시간 상관 분석**: 감쇠 진동의 A-C 함수 피크 높이 분석
8. **종합 시각화**: Supergranulation + 5분 진동 통합 속도장

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.ndimage import gaussian_filter
from scipy.signal import correlate2d, correlate

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Physical constants
R_SUN = 6.96e5  # km, solar radius
C_LIGHT = 3e5   # km/s, speed of light
G_SUN = 0.274   # km/s^2, solar surface gravity

## Part 1: Line-Shifter 시뮬레이션 / Line-Shifter Simulation

Leighton의 line-shifter는 흡수선의 적색/청색 날개를 동시에 관측합니다.
Doppler shift $v$가 있으면 선 프로파일이 이동하여 양쪽 날개의 밝기가 반대로 변합니다:

Leighton's line-shifter simultaneously observes the red and violet wings of an absorption line.
A Doppler shift $v$ displaces the line profile, causing opposite brightness changes in the two wings:

$$I_r = I_0[1 + \delta(x,y) + \beta(x,y)], \quad I_v = I_0[1 - \delta(x,y) + \beta(x,y)]$$

여기서 $\delta \propto v/c$ (Doppler 성분), $\beta$ (고유 밝기 변동)
where $\delta \propto v/c$ (Doppler component), $\beta$ (intrinsic brightness variation)

In [ ]:
# Simulate a Gaussian absorption line and the line-shifter effect
# 가우시안 흡수선과 line-shifter 효과 시뮬레이션

def gaussian_line(wavelength, center, width, depth):
    """Model a Gaussian absorption line profile."""
    return 1.0 - depth * np.exp(-0.5 * ((wavelength - center) / width)**2)

# Line parameters (Ca 6103 Å)
lambda_0 = 6103.0  # Å, line center
line_width = 0.15  # Å, Gaussian sigma
line_depth = 0.7   # fractional depth
slit_offset = 0.10  # Å, line-shifter offset from center

wavelength = np.linspace(6102.0, 6104.0, 1000)
profile = gaussian_line(wavelength, lambda_0, line_width, line_depth)

# Slit positions for red and violet wings
red_slit = lambda_0 + slit_offset
vio_slit = lambda_0 - slit_offset

# Simulate Doppler shift: v = 0.5 km/s → Δλ = v/c * λ₀
v_doppler = 0.5  # km/s
delta_lambda = v_doppler / C_LIGHT * lambda_0  # ~0.010 Å

# Shifted profile
profile_shifted = gaussian_line(wavelength, lambda_0 + delta_lambda,
                                line_width, line_depth)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: line profile with slit positions
ax = axes[0]
ax.plot(wavelength, profile, 'b-', lw=2, label='Rest profile / 정지 프로파일')
ax.plot(wavelength, profile_shifted, 'r--', lw=2,
        label=f'Shifted (v={v_doppler} km/s)')
ax.axvline(red_slit, color='red', ls=':', alpha=0.7, label=f'Red slit ({red_slit} Å)')
ax.axvline(vio_slit, color='blue', ls=':', alpha=0.7, label=f'Violet slit ({vio_slit} Å)')
ax.axvline(lambda_0, color='gray', ls='-', alpha=0.3)
ax.set_xlabel('Wavelength / 파장 (Å)')
ax.set_ylabel('Intensity / 강도 (normalized)')
ax.set_title('Line-Shifter: Absorption Line & Slit Positions\n흡수선과 슬릿 위치')
ax.legend(fontsize=9)

# Mark intensity at slit positions
I_rest_red = gaussian_line(red_slit, lambda_0, line_width, line_depth)
I_rest_vio = gaussian_line(vio_slit, lambda_0, line_width, line_depth)
I_shift_red = gaussian_line(red_slit, lambda_0 + delta_lambda, line_width, line_depth)
I_shift_vio = gaussian_line(vio_slit, lambda_0 + delta_lambda, line_width, line_depth)

ax.plot(red_slit, I_rest_red, 'ro', ms=8)
ax.plot(vio_slit, I_rest_vio, 'bo', ms=8)
ax.plot(red_slit, I_shift_red, 'r^', ms=10)
ax.plot(vio_slit, I_shift_vio, 'b^', ms=10)

# Right: intensity difference as function of velocity
velocities = np.linspace(-2, 2, 200)  # km/s
I_red = []
I_vio = []
for v in velocities:
    dl = v / C_LIGHT * lambda_0
    I_red.append(gaussian_line(red_slit, lambda_0 + dl, line_width, line_depth))
    I_vio.append(gaussian_line(vio_slit, lambda_0 + dl, line_width, line_depth))

I_red = np.array(I_red)
I_vio = np.array(I_vio)

ax = axes[1]
ax.plot(velocities, I_red, 'r-', lw=2, label='Red wing intensity / 적색 날개')
ax.plot(velocities, I_vio, 'b-', lw=2, label='Violet wing intensity / 청색 날개')
ax.plot(velocities, I_red - I_vio, 'k--', lw=2, label='Difference (Doppler signal)')
ax.axhline(0, color='gray', ls='-', alpha=0.3)
ax.axvline(0, color='gray', ls='-', alpha=0.3)
ax.set_xlabel('Line-of-sight velocity / 시선 속도 (km/s)')
ax.set_ylabel('Intensity / 강도')
ax.set_title('Doppler Signal: Red - Violet Wing\nDoppler 신호: 적색 - 청색 날개 차이')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Doppler shift for v = {v_doppler} km/s: Δλ = {delta_lambda:.4f} Å")
print(f"  Red wing: I_rest = {I_rest_red:.4f}, I_shifted = {I_shift_red:.4f}, "
      f"ΔI = {I_shift_red - I_rest_red:+.4f}")
print(f"  Violet wing: I_rest = {I_rest_vio:.4f}, I_shifted = {I_shift_vio:.4f}, "
      f"ΔI = {I_shift_vio - I_rest_vio:+.4f}")

## Part 2: Supergranulation 속도장 생성 / Supergranulation Velocity Field

Supergranulation은 태양 표면 전체에 ~5,000개의 대류 셀이 분포하며,
각 셀 내부에서 중심 → 경계로 향하는 수평 유출 흐름이 존재합니다.

Supergranulation consists of ~5,000 convection cells distributed across the solar surface,
with horizontal outflow from the center to the boundary within each cell.

셀 내 수평 속도: $V_\rho = F(\rho)$ (셀 중심에서 0, 경계에서 최대)
시선 속도 투영: $V_1(x,y) = \frac{1-\mu^2}{\mu} \frac{y}{\rho} F(\rho)$

Horizontal velocity in a cell: $V_\rho = F(\rho)$ (zero at center, max at boundary)
Line-of-sight projection: $V_1(x,y) = \frac{1-\mu^2}{\mu} \frac{y}{\rho} F(\rho)$

In [ ]:
# Generate a supergranulation velocity field using Voronoi-like cells
# Voronoi 형태의 supergranulation 속도장 생성

np.random.seed(42)

# Grid: 200,000 km × 200,000 km patch of the solar surface
grid_size = 200_000  # km
N = 512
x = np.linspace(0, grid_size, N)
y = np.linspace(0, grid_size, N)
X, Y = np.meshgrid(x, y)

# Place ~40 cell centers (average spacing ~30,000 km)
n_cells = 40
cell_x = np.random.uniform(0, grid_size, n_cells)
cell_y = np.random.uniform(0, grid_size, n_cells)

# For each point, find the nearest cell center and compute flow
Vx = np.zeros_like(X)
Vy = np.zeros_like(Y)

# Characteristic cell radius
R_cell = 15_000  # km, half of 30,000 km diameter
V_max = 0.5  # km/s, maximum horizontal velocity

for cx, cy in zip(cell_x, cell_y):
    dx = X - cx
    dy = Y - cy
    rho = np.sqrt(dx**2 + dy**2)
    
    # Radial velocity profile: F(rho) = V_max * (rho/R) * exp(1 - rho/R)
    # Peaks at rho = R_cell, goes to 0 at center
    F = V_max * (rho / R_cell) * np.exp(1 - rho / R_cell)
    
    # Mask to influence region (within ~2 cell radii)
    mask = rho < 2.5 * R_cell
    
    # Unit radial direction
    with np.errstate(divide='ignore', invalid='ignore'):
        ux = np.where(rho > 0, dx / rho, 0)
        uy = np.where(rho > 0, dy / rho, 0)
    
    # Add contribution (weighted by proximity)
    weight = np.exp(-0.5 * (rho / R_cell)**2)
    Vx += F * ux * weight
    Vy += F * uy * weight

# Normalize to realistic amplitude
speed = np.sqrt(Vx**2 + Vy**2)
Vx = Vx / (speed.max() + 1e-10) * V_max
Vy = Vy / (speed.max() + 1e-10) * V_max
speed = np.sqrt(Vx**2 + Vy**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: horizontal velocity magnitude
ax = axes[0]
im = ax.pcolormesh(X / 1000, Y / 1000, speed, cmap='hot', shading='auto')
plt.colorbar(im, ax=ax, label='|V_h| (km/s)')
# Plot cell centers
ax.plot(cell_x / 1000, cell_y / 1000, 'c+', ms=8, mew=2)
# Quiver plot (subsample)
skip = 20
ax.quiver(X[::skip, ::skip] / 1000, Y[::skip, ::skip] / 1000,
          Vx[::skip, ::skip], Vy[::skip, ::skip],
          color='cyan', alpha=0.5, scale=5)
ax.set_xlabel('x (×10³ km)')
ax.set_ylabel('y (×10³ km)')
ax.set_title('Supergranulation: Horizontal Velocity Field\n수평 속도장 (상면도 / top view)')
ax.set_aspect('equal')

# Right: line-of-sight projection at different disk positions (mu values)
# Assume the y-direction is parallel to the limb direction
ax = axes[1]
mu_values = [0.95, 0.7, 0.4]  # cos(theta) = disk center to limb

for mu in mu_values:
    # V_los = sqrt(1-mu^2) * Vy (simplified projection for y-direction)
    V_los = np.sqrt(1 - mu**2) * Vy
    
    # Take a 1D slice
    mid = N // 2
    ax.plot(x / 1000, V_los[mid, :],
            label=f'μ = {mu:.2f} (θ = {np.degrees(np.arccos(mu)):.0f}°)', lw=1.5)

ax.set_xlabel('x (×10³ km)')
ax.set_ylabel('V_LOS (km/s)')
ax.set_title('Line-of-Sight Velocity vs. Disk Position\n시선 속도 — disk 위치별 투영')
ax.legend()
ax.axhline(0, color='gray', ls='-', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Supergranulation field: {n_cells} cells in {grid_size/1000:.0f}×{grid_size/1000:.0f} "
      f"(×10³ km)² area")
print(f"Average cell spacing: ~{grid_size / np.sqrt(n_cells) / 1000:.0f} ×10³ km")
print(f"Max horizontal velocity: {speed.max():.2f} km/s")

## Part 3: 자기상관(A-C) 함수 분석 / Autocorrelation Function Analysis

논문에서 핵심 정량 도구인 A-C 함수를 구현합니다.
A-C 함수의 FWHM은 패턴의 특성 크기를, 2차 극대는 셀 간격의 규칙성을 나타냅니다.

We implement the A-C function, the key quantitative tool in the paper.
The FWHM of the A-C function gives the characteristic size, and secondary maxima indicate regularity in cell spacing.

$$C(s) = \frac{K}{A}\int\int T(x,y) \, T(x+s, y) \, dA$$

In [ ]:
# Compute 1D autocorrelation function of the velocity field
# 속도장의 1D 자기상관 함수 계산

def compute_ac_1d(signal):
    """Compute normalized 1D autocorrelation function."""
    n = len(signal)
    mean = np.mean(signal)
    var = np.var(signal)
    if var == 0:
        return np.zeros(n)
    ac = correlate(signal - mean, signal - mean, mode='full')
    # Normalize
    ac = ac[n-1:]  # take positive lags only
    ac = ac / ac[0]
    return ac

# Use V_los at mu = 0.5 (limb region where supergranulation is visible)
mu = 0.5
V_los_field = np.sqrt(1 - mu**2) * Vy

# Take a 1D slice through the middle
mid_slice = V_los_field[N // 2, :]
ac = compute_ac_1d(mid_slice)

# Compute lag in km
dx = grid_size / N  # km per pixel
lags = np.arange(len(ac)) * dx

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: the velocity slice
ax = axes[0]
ax.plot(x / 1000, mid_slice, 'b-', lw=1)
ax.set_xlabel('Position / 위치 (×10³ km)')
ax.set_ylabel('V_LOS (km/s)')
ax.set_title('1D Velocity Slice (μ = 0.5)\n1D 속도 프로파일')
ax.axhline(0, color='gray', ls='-', alpha=0.3)

# Right: autocorrelation function
ax = axes[1]
max_lag = 100_000  # km
mask = lags < max_lag
ax.plot(lags[mask] / 1000, ac[mask], 'r-', lw=2)
ax.axhline(0, color='gray', ls='-', alpha=0.3)

# Find FWHM
half_max = 0.5
idx_half = np.where(ac[:sum(mask)] < half_max)[0]
if len(idx_half) > 0:
    fwhm_km = lags[idx_half[0]]
    ax.axvline(fwhm_km / 1000, color='blue', ls='--', alpha=0.7,
               label=f'FWHM ≈ {fwhm_km/1000:.0f} ×10³ km')

# Find secondary maximum (undershoot then rise)
ac_positive = ac[:sum(mask)]
sign_changes = np.where(np.diff(np.sign(ac_positive)))[0]
if len(sign_changes) >= 2:
    # Look for secondary max between second and third zero-crossing
    region = ac_positive[sign_changes[1]:]
    sec_max_idx = sign_changes[1] + np.argmax(region)
    sec_max_lag = lags[sec_max_idx]
    ax.axvline(sec_max_lag / 1000, color='green', ls='--', alpha=0.7,
               label=f'2nd max ≈ {sec_max_lag/1000:.0f} ×10³ km')

ax.set_xlabel('Lag / 지연 (×10³ km)')
ax.set_ylabel('Autocorrelation / 자기상관')
ax.set_title('A-C Function of Supergranulation\nSupergranulation의 A-C 함수 (cf. Fig. 7)')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Compare with paper: FWHM = 14,000–18,000 km, "
      f"secondary max at ~35,000 km")

## Part 4: 밝기–속도 상관 분석 / Brightness–Velocity Correlation

대류에서 밝은(뜨거운) 요소는 상승하고 어두운(차가운) 요소는 하강합니다.
이 상관관계를 적색/청색 날개 이미지의 A-C 함수로부터 추출할 수 있습니다.

In convection, bright (hot) elements rise and dark (cool) elements sink.
This correlation can be extracted from the A-C functions of red/violet wing images.

$$C = \frac{\langle\beta \, v'\rangle}{\langle\beta^2\rangle^{1/2} \langle v'^2\rangle^{1/2}}$$

핵심 발견: 저층에서 $C > 0$, 고층에서 $C < 0$ — 고도에 따른 부호 반전!
Key finding: $C > 0$ at low layers, $C < 0$ at high layers — sign reversal with altitude!

In [ ]:
# Simulate brightness-velocity correlation at different atmospheric heights
# 대기 고도별 밝기–속도 상관관계 시뮬레이션

np.random.seed(123)
n_elements = 5000  # number of convective elements

# Generate convective velocity field: upflows and downflows
v_vertical = np.random.normal(0, 0.4, n_elements)  # km/s, rms ~ 0.4

# Temperature fluctuation model:
# At the photosphere: hot rises (ΔT > 0 when v > 0)
# At the chromosphere: rising material has cooled (ΔT < 0 when v > 0)

def brightness_at_height(velocity, height_km, noise_level=0.1):
    """Model brightness fluctuation β as function of vertical velocity and height.
    
    At low heights: β ∝ +v (hot material rises → bright)
    At high heights: β ∝ -v (adiabatic cooling → rising material is dark)
    Transition around 500 km above photosphere.
    """
    # Correlation coefficient varies with height
    # Positive below ~500 km, negative above
    transition_height = 500  # km
    scale_height = 200  # km
    
    # Smooth transition from +1 to -1
    correlation_sign = -np.tanh((height_km - transition_height) / scale_height)
    
    # β = correlation_sign * |correlation| * v + noise
    beta = correlation_sign * 0.5 * velocity + np.random.normal(0, noise_level, len(velocity))
    return beta, correlation_sign

# Compute at different heights (corresponding to different spectral lines)
heights = {
    'Fe 6102 (low photosphere)': 100,
    'Ca 6103 (mid photosphere)': 200,
    'Ba⁺ 4554 (photosphere)': 250,
    'Na 5896 wing (upper photosphere)': 400,
    'Na 5896 core (low chromosphere)': 700,
    'Ca⁺ 8542 (chromosphere)': 1000,
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

correlations = {}
for idx, (line_name, h) in enumerate(heights.items()):
    beta, corr_sign = brightness_at_height(v_vertical, h)
    
    # Compute correlation coefficient
    beta_prime = beta - np.mean(beta)
    v_prime = v_vertical - np.mean(v_vertical)
    C = np.mean(beta_prime * v_prime) / (np.std(beta) * np.std(v_vertical))
    correlations[line_name] = C
    
    ax = axes[idx]
    ax.scatter(v_vertical, beta, alpha=0.1, s=1, c='steelblue')
    
    # Linear fit for visualization
    coeffs = np.polyfit(v_vertical, beta, 1)
    v_fit = np.linspace(-1.5, 1.5, 100)
    ax.plot(v_fit, np.polyval(coeffs, v_fit), 'r-', lw=2)
    
    ax.set_xlabel('Vertical velocity (km/s)')
    ax.set_ylabel('Brightness β')
    ax.set_title(f'{line_name}\nh = {h} km, C = {C:+.2f}')
    ax.axhline(0, color='gray', ls='-', alpha=0.3)
    ax.axvline(0, color='gray', ls='-', alpha=0.3)
    ax.set_xlim(-1.5, 1.5)

plt.suptitle('Brightness–Velocity Correlation at Different Heights\n'
             '고도별 밝기–속도 상관관계 (cf. Table 1)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print("\n=== Brightness-Velocity Correlation Summary / 밝기-속도 상관 요약 ===")
print(f"{'Line':<40} {'Height (km)':<15} {'C (simulated)':<15} {'C (paper)'}")
print("-" * 90)
paper_C = ['+0.38', '+0.50', '+0.38', '+0.12', '-0.23', '-0.12']
for (name, h), pc in zip(heights.items(), paper_C):
    print(f"{name:<40} {h:<15} {correlations[name]:+.2f}{'':>10} {pc}")

## Part 5: 5분 진동 시뮬레이션 / 5-Minute Oscillation Simulation

논문의 가장 놀라운 발견: 태양 표면의 수직 속도가 $T = 296$ sec 주기의 감쇠 진동을 보입니다.

The most striking discovery: vertical velocities on the solar surface show damped oscillations with $T = 296$ sec.

감쇠 진동 모델:
Damped oscillation model:

$$v(x, y, t) = A(x,y) \, e^{-t/\tau} \cos(\omega t + \phi(x,y))$$

여기서 $\omega = 2\pi/T$, $\tau \sim 380$ sec (평균 수명 / mean life)

In [ ]:
# Simulate the 5-minute oscillation and Doppler difference plates
# 5분 진동 시뮬레이션 및 Doppler difference plate

T_osc = 296.0  # sec, oscillation period
omega = 2 * np.pi / T_osc
tau = 380.0  # sec, mean lifetime
v_rms = 0.4  # km/s

# Time array
t = np.linspace(0, 1500, 1000)  # sec

# Single oscillating element
v_single = v_rms * np.sqrt(2) * np.exp(-t / tau) * np.cos(omega * t)

# Ensemble of oscillating elements with random phases and start times
np.random.seed(77)
n_osc = 200
phases = np.random.uniform(0, 2 * np.pi, n_osc)
start_times = np.random.exponential(tau, n_osc)  # elements starting at different times
amplitudes = np.random.rayleigh(v_rms, n_osc)

# Composite velocity at a single point (sum of many oscillators)
v_composite = np.zeros_like(t)
for phi, t0, amp in zip(phases, start_times, amplitudes):
    active = t > t0
    v_composite[active] += amp * np.exp(-(t[active] - t0) / tau) * np.cos(omega * (t[active] - t0) + phi)

v_composite /= np.sqrt(n_osc / 10)  # normalize to reasonable amplitude

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: single element oscillation
ax = axes[0, 0]
ax.plot(t, v_single, 'b-', lw=1.5)
ax.plot(t, v_rms * np.sqrt(2) * np.exp(-t / tau), 'r--', lw=1, alpha=0.7,
        label=f'Envelope: exp(-t/{tau:.0f}s)')
ax.plot(t, -v_rms * np.sqrt(2) * np.exp(-t / tau), 'r--', lw=1, alpha=0.7)
ax.set_xlabel('Time / 시간 (sec)')
ax.set_ylabel('Velocity / 속도 (km/s)')
ax.set_title('Single Oscillating Element\n단일 진동 요소')
ax.legend()

# Top-right: composite signal at a point
ax = axes[0, 1]
ax.plot(t, v_composite, 'darkblue', lw=0.8)
ax.set_xlabel('Time / 시간 (sec)')
ax.set_ylabel('Velocity / 속도 (km/s)')
ax.set_title('Composite Velocity at One Point\n한 지점에서의 합성 속도')

# Bottom-left: Doppler difference plate simulation
# At each spatial point, velocity oscillates with random phase
# Difference plate shows v(x, t1) - v(x, t2)
ax = axes[1, 0]
n_spatial = 500
spatial_phases = np.random.uniform(0, 2 * np.pi, n_spatial)
spatial_amps = np.random.rayleigh(v_rms, n_spatial)

dt_values = [0, 75, 148, 222, 296, 370, 444]  # sec
colors = plt.cm.viridis(np.linspace(0, 1, len(dt_values)))

x_spatial = np.arange(n_spatial)
for dt_val, color in zip(dt_values, colors):
    v_t0 = spatial_amps * np.cos(spatial_phases)
    v_t1 = spatial_amps * np.exp(-dt_val / tau) * np.cos(omega * dt_val + spatial_phases)
    diff = v_t1 - v_t0
    rms_diff = np.std(diff)
    ax.plot(dt_val, rms_diff, 'o', color=color, ms=10)

# Compute full curve
dt_full = np.linspace(0, 800, 200)
rms_theory = []
for dt_val in dt_full:
    v_t0 = spatial_amps * np.cos(spatial_phases)
    v_t1 = spatial_amps * np.exp(-dt_val / tau) * np.cos(omega * dt_val + spatial_phases)
    rms_theory.append(np.std(v_t1 - v_t0))

ax.plot(dt_full, rms_theory, 'b-', lw=2)
ax.axvline(T_osc / 2, color='red', ls=':', alpha=0.7, label=f'T/2 = {T_osc/2:.0f} sec')
ax.axvline(T_osc, color='green', ls=':', alpha=0.7, label=f'T = {T_osc:.0f} sec')
ax.set_xlabel('Time delay Δt / 시간 지연 (sec)')
ax.set_ylabel('RMS of velocity difference\n속도 차이의 RMS (km/s)')
ax.set_title('Doppler Difference Plate: Signal vs. Δt\nDoppler 차이 신호 vs. 시간 지연 (cf. Fig. 15)')
ax.legend()

# Bottom-right: Visualization of expected vs observed
ax = axes[1, 1]
# Expected (turbulent): monotonic increase
dt_plot = np.linspace(0, 800, 100)
turbulent_rms = v_rms * np.sqrt(2) * np.sqrt(1 - np.exp(-dt_plot / 200))
ax.plot(dt_plot, turbulent_rms, 'gray', ls='--', lw=2,
        label='Expected (turbulent)\n예상 (난류)')

# Observed (oscillatory): periodic maxima/minima
osc_rms = v_rms * np.sqrt(2) * np.sqrt(
    1 - np.exp(-dt_plot / tau) * np.cos(omega * dt_plot))
ax.plot(dt_plot, osc_rms, 'b-', lw=2,
        label='Observed (oscillatory)\n관측 (진동)')

ax.axvline(T_osc / 2, color='red', ls=':', alpha=0.5)
ax.axvline(T_osc, color='green', ls=':', alpha=0.5)
ax.set_xlabel('Time delay Δt / 시간 지연 (sec)')
ax.set_ylabel('RMS velocity change\n속도 변화 RMS (km/s)')
ax.set_title('Expected vs. Observed Velocity Change\n예상 vs. 관측된 속도 변화')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"5-minute oscillation parameters:")
print(f"  Period T = {T_osc} sec = {T_osc/60:.1f} min")
print(f"  Mean life τ = {tau} sec = {tau/60:.1f} min")
print(f"  τ/T = {tau/T_osc:.2f} (oscillation persists ~{tau/T_osc:.1f} cycles)")
print(f"  Angular frequency ω = {omega:.4f} rad/s")
print(f"  RMS velocity ~ {v_rms} km/s")

## Part 6: 에너지 수송 추정 / Mechanical Energy Transport Estimate

대류에 의한 비복사 에너지 수송:
Non-radiative energy transport by convection:

$$\langle E \rangle = \frac{\gamma}{\gamma - 1} P_0 \left\langle \frac{\Delta T}{T} v \right\rangle$$

여기서 $\gamma = 5/3$ (단원자 이상 기체), $P_0$는 평균 압력, $\Delta T/T$는 상대 온도 변동, $v$는 수직 속도.
where $\gamma = 5/3$ (monatomic ideal gas), $P_0$ is mean pressure, $\Delta T/T$ is relative temperature fluctuation, $v$ is vertical velocity.

In [ ]:
# Estimate mechanical energy transport from convective motions
# 대류 운동에 의한 기계적 에너지 수송 추정

gamma = 5.0 / 3.0  # adiabatic index for monatomic gas

# Photospheric parameters at Fe 6102 / Ca 6103 formation height
P0_dyn = 4e4  # dyn/cm^2, gas pressure at photospheric line formation
P0_Pa = P0_dyn * 0.1  # Pa

# From Table 1: <βv> values for each line
# β = a * ΔT/T where a ~ 4 for weak lines
a = 4.0
lines_data = {
    'Fe 6102': {'beta_v': 0.0057, 'v_rms': 0.39, 'C': 0.38,
                'P0': 4e4, 'height': 'Low photosphere'},
    'Ca 6103': {'beta_v': 0.0103, 'v_rms': 0.43, 'C': 0.50,
                'P0': 4e4, 'height': 'Mid photosphere'},
    'Ba⁺ 4554': {'beta_v': 0.0166, 'v_rms': 0.61, 'C': 0.38,
                 'P0': 3e4, 'height': 'Photosphere'},
    'Na 5896 (wing)': {'beta_v': 0.0022, 'v_rms': 0.35, 'C': 0.12,
                       'P0': 2e4, 'height': 'Upper photosphere'},
    'Na 5896 (core)': {'beta_v': -0.0051, 'v_rms': 0.50, 'C': -0.23,
                       'P0': 1e4, 'height': 'Low chromosphere'},
}

print("=== Mechanical Energy Transport Estimates / 기계적 에너지 수송 추정 ===\n")
print(f"{'Line':<20} {'<βv>':<10} {'<ΔT/T · v>':<12} {'P₀ (dyn/cm²)':<14} "
      f"{'<E> (W/cm²)':<12} {'Height'}")
print("-" * 85)

energies = {}
for line, data in lines_data.items():
    # <ΔT/T · v> = <βv> / a
    dt_t_v = data['beta_v'] / a  # (km/s)
    dt_t_v_cgs = dt_t_v * 1e5  # cm/s
    
    # <E> = γ/(γ-1) * P₀ * <ΔT/T · v>
    E_cgs = gamma / (gamma - 1) * data['P0'] * dt_t_v_cgs  # erg/cm²/s
    E_W = E_cgs * 1e-7 * 1e4  # W/m²
    E_Wcm2 = E_cgs * 1e-7  # W/cm²
    
    energies[line] = E_Wcm2
    print(f"{line:<20} {data['beta_v']:<10.4f} {dt_t_v:<12.5f} {data['P0']:<14.0f} "
          f"{E_Wcm2:<12.2f} {data['height']}")

print(f"\n{'Paper estimate / 논문 추정':>50}: ~2 W/cm² at Fe 6102 / Ca 6103 level")
print(f"{'Chromospheric heating requirement / 색구 가열 필요량':>50}: ~1-4 W/cm²")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))

heights_km = [100, 200, 250, 400, 700]
E_values = [energies[k] for k in lines_data.keys()]
line_names = list(lines_data.keys())

colors = ['steelblue' if e > 0 else 'coral' for e in E_values]
bars = ax.barh(line_names, E_values, color=colors, edgecolor='black', alpha=0.8)
ax.axvline(0, color='black', lw=0.5)
ax.axvline(2, color='red', ls='--', alpha=0.5, label='Paper estimate (~2 W/cm²)')
ax.set_xlabel('Energy flux <E> (W/cm²) / 에너지 플럭스')
ax.set_title('Mechanical Energy Transport at Different Heights\n고도별 기계적 에너지 수송 (cf. Eq. 9)')
ax.legend()

plt.tight_layout()
plt.show()

## Part 7: 시간 상관 분석 — A-C 피크 높이 / Temporal Correlation — A-C Peak Heights

논문의 Fig. 21과 22를 재현합니다. Doppler difference plate의 A-C 피크 높이를
시간 지연 $\Delta t$의 함수로 그리면, 진동 주기성과 감쇠를 정량적으로 측정할 수 있습니다.

We reproduce Figs. 21 and 22 from the paper. Plotting A-C peak heights of Doppler difference plates as a function of time delay $\Delta t$ enables quantitative measurement of oscillation periodicity and damping.

$$H(\Delta t) = A \, e^{-\Delta t / \tau} (1 - \cos \, n\omega T)$$

In [ ]:
# Reproduce Fig. 21 and Fig. 22 from the paper
# 논문의 Fig. 21과 Fig. 22 재현

T = 296.0  # sec
omega = 2 * np.pi / T
tau = 380.0  # sec

# Time delay in units of T
dt_T = np.linspace(0, 4, 200)
dt = dt_T * T

# A-C peak height model: H(Δt) ∝ (1 - exp(-Δt/τ) cos(ωΔt))
# This is proportional to the mean-square velocity difference
# H(Δt) = 2<v²> - 2<v(t)v(t+Δt)>
# For damped oscillation: <v(t)v(t+Δt)> = <v²> exp(-Δt/τ) cos(ωΔt)

def H_model(dt, v_rms, tau, omega):
    """A-C peak height for damped oscillatory field."""
    return v_rms**2 * (1 - np.exp(-dt / tau) * np.cos(omega * dt))

# Different spectral lines with different rms velocities (from Table 2)
lines = {
    'Fe 6102': {'v_rms': 0.46, 'color': 'blue', 'marker': '^'},
    'Ca 6103': {'v_rms': 0.47, 'color': 'red', 'marker': 'o'},
    'Na 5896': {'v_rms': 0.56, 'color': 'green', 'marker': 's'},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Fig. 21 — A-C peak heights vs Δt/T
ax = axes[0]
for name, data in lines.items():
    H = H_model(dt, data['v_rms'], tau, omega)
    # Normalize to show relative variation
    H_norm = H / data['v_rms']**2
    ax.plot(dt_T, H_norm, color=data['color'], lw=2, label=name)

# Add markers at integer and half-integer periods
for n in range(5):
    ax.axvline(n, color='gray', ls=':', alpha=0.3)
    ax.axvline(n + 0.5, color='gray', ls=':', alpha=0.15)

ax.set_xlabel('Δt / T')
ax.set_ylabel('H / <v²> (normalized peak height)')
ax.set_title('A-C Peak Height vs. Δt/T\nA-C 피크 높이 vs. Δt/T (cf. Fig. 21)')
ax.legend()
ax.set_ylim(0, 2.5)

# Right: Fig. 22 — Semilog plot of successive peak differences
# |H(nT) - H((n-1/2)T)| vs Δt/T → measures damping
ax = axes[1]

n_vals = np.arange(0.5, 4, 0.5)
for name, data in lines.items():
    dt_points = n_vals * T
    H_points = H_model(dt_points, data['v_rms'], tau, omega)
    
    # Successive differences at integer T vs half-integer T
    D = []
    dt_D = []
    for i in range(1, len(n_vals)):
        d = abs(H_points[i] - H_points[i-1])
        if d > 0:
            D.append(d)
            dt_D.append(n_vals[i])
    
    ax.semilogy(dt_D, D, marker=data['marker'], color=data['color'],
                label=name, lw=1.5, ms=8)

# Theoretical decay lines
dt_theory = np.linspace(0.5, 3.5, 100)
for tau_val, ls in [(320, '--'), (440, ':')]:
    decay = 0.15 * np.exp(-dt_theory * T / tau_val)
    ax.semilogy(dt_theory, decay, color='gray', ls=ls,
                label=f'τ = {tau_val} sec')

ax.set_xlabel('Δt / T')
ax.set_ylabel('|H(nT) - H((n-½)T)| (log scale)')
ax.set_title('Damping Rate: Successive Peak Differences\n감쇠율: 연속 피크 차이 (cf. Fig. 22)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"From paper: τ ~ 380 sec (range 320-440 sec from two fitted lines)")
print(f"  → Oscillation persists for τ/T = {380/296:.1f} cycles before decaying to 1/e")

## Part 8: 종합 시각화 — 통합 속도장 / Combined Visualization — Integrated Velocity Field

Supergranulation의 수평 흐름과 5분 진동의 수직 운동을 함께 시각화합니다.
실제 태양에서는 이 두 가지가 동시에 존재하지만, 서로 다른 스케일과 시간을 가집니다.

We visualize both the horizontal supergranulation flow and the vertical 5-minute oscillations together.
On the real Sun, both coexist but operate at different spatial and temporal scales.

| 특성 / Property | Supergranulation | 5-min Oscillations |
|---|---|---|
| 방향 / Direction | 수평 / Horizontal | 수직 / Vertical |
| 크기 / Size | ~30,000 km | ~2,000–3,000 km |
| 시간 / Timescale | 수 시간 / Hours | ~5 min |
| 속도 / Velocity | ~0.5 km/s | ~0.4 km/s rms |

In [ ]:
# Combined visualization: supergranulation + 5-minute oscillations
# 종합 시각화: supergranulation + 5분 진동

np.random.seed(42)

# Create a smaller grid for detailed view
N2 = 256
grid2 = 100_000  # 100,000 km
x2 = np.linspace(0, grid2, N2)
y2 = np.linspace(0, grid2, N2)
X2, Y2 = np.meshgrid(x2, y2)
dx2 = grid2 / N2

# Supergranulation: ~10 cells in this area
n_cells2 = 10
cx2 = np.random.uniform(0, grid2, n_cells2)
cy2 = np.random.uniform(0, grid2, n_cells2)

Vx2 = np.zeros((N2, N2))
Vy2 = np.zeros((N2, N2))

for cx, cy in zip(cx2, cy2):
    ddx = X2 - cx
    ddy = Y2 - cy
    rho = np.sqrt(ddx**2 + ddy**2)
    F = V_max * (rho / R_cell) * np.exp(1 - rho / R_cell)
    weight = np.exp(-0.5 * (rho / R_cell)**2)
    with np.errstate(divide='ignore', invalid='ignore'):
        ux = np.where(rho > 0, ddx / rho, 0)
        uy = np.where(rho > 0, ddy / rho, 0)
    Vx2 += F * ux * weight
    Vy2 += F * uy * weight

speed2 = np.sqrt(Vx2**2 + Vy2**2)
Vx2 = Vx2 / (speed2.max() + 1e-10) * 0.5
Vy2 = Vy2 / (speed2.max() + 1e-10) * 0.5

# 5-minute oscillations: small-scale vertical velocity
# Generate oscillating elements with random phases
n_osc_elements = 200
osc_x = np.random.uniform(0, grid2, n_osc_elements)
osc_y = np.random.uniform(0, grid2, n_osc_elements)
osc_phase = np.random.uniform(0, 2 * np.pi, n_osc_elements)
osc_amp = np.random.rayleigh(0.3, n_osc_elements)
osc_size = np.random.uniform(2000, 4000, n_osc_elements)  # km, element size

# Vertical velocity field at t=0
Vz = np.zeros((N2, N2))
for ox, oy, ophi, oamp, osize in zip(osc_x, osc_y, osc_phase, osc_amp, osc_size):
    r = np.sqrt((X2 - ox)**2 + (Y2 - oy)**2)
    Vz += oamp * np.cos(ophi) * np.exp(-0.5 * (r / osize)**2)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Top-left: supergranulation horizontal flow only
ax = axes[0, 0]
speed_plot = np.sqrt(Vx2**2 + Vy2**2)
im = ax.pcolormesh(X2/1000, Y2/1000, speed_plot, cmap='YlOrRd', shading='auto')
plt.colorbar(im, ax=ax, label='|V_h| (km/s)')
skip2 = 12
ax.quiver(X2[::skip2, ::skip2]/1000, Y2[::skip2, ::skip2]/1000,
          Vx2[::skip2, ::skip2], Vy2[::skip2, ::skip2],
          color='black', alpha=0.6, scale=5)
ax.plot(cx2/1000, cy2/1000, 'k+', ms=12, mew=2)
ax.set_title('Supergranulation Only (horizontal)\nSupergranulation만 (수평)')
ax.set_xlabel('x (×10³ km)')
ax.set_ylabel('y (×10³ km)')
ax.set_aspect('equal')

# Top-right: 5-min oscillation vertical field only
ax = axes[0, 1]
vlim = np.percentile(np.abs(Vz), 98)
norm = TwoSlopeNorm(vmin=-vlim, vcenter=0, vmax=vlim)
im = ax.pcolormesh(X2/1000, Y2/1000, Vz, cmap='bwr', norm=norm, shading='auto')
plt.colorbar(im, ax=ax, label='V_z (km/s)')
ax.set_title('5-min Oscillations Only (vertical, t=0)\n5분 진동만 (수직, t=0)')
ax.set_xlabel('x (×10³ km)')
ax.set_ylabel('y (×10³ km)')
ax.set_aspect('equal')

# Bottom-left: what a Doppler plate would see (combined, at disk center: V_z only)
ax = axes[1, 0]
im = ax.pcolormesh(X2/1000, Y2/1000, Vz, cmap='bwr', norm=norm, shading='auto')
plt.colorbar(im, ax=ax, label='V_LOS (km/s)')
ax.set_title('Doppler Plate at Disk Center (μ=1)\n태양 중심 Doppler plate — 수직 운동만 보임')
ax.set_xlabel('x (×10³ km)')
ax.set_ylabel('y (×10³ km)')
ax.set_aspect('equal')

# Bottom-right: what a Doppler plate would see (combined, near limb: V_h dominates)
mu_limb = 0.4
V_los_limb = np.sqrt(1 - mu_limb**2) * Vy2 + mu_limb * Vz
ax = axes[1, 1]
vlim2 = np.percentile(np.abs(V_los_limb), 98)
norm2 = TwoSlopeNorm(vmin=-vlim2, vcenter=0, vmax=vlim2)
im = ax.pcolormesh(X2/1000, Y2/1000, V_los_limb, cmap='bwr', norm=norm2, shading='auto')
plt.colorbar(im, ax=ax, label='V_LOS (km/s)')
ax.plot(cx2/1000, cy2/1000, 'k+', ms=10, mew=1.5, alpha=0.5)
ax.set_title(f'Doppler Plate Near Limb (μ={mu_limb})\n림 근처 Doppler plate — 수평+수직 혼합')
ax.set_xlabel('x (×10³ km)')
ax.set_ylabel('y (×10³ km)')
ax.set_aspect('equal')

plt.suptitle('Two Velocity Fields on the Solar Surface\n태양 표면의 두 속도장',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Key insight: At disk center (μ=1), only vertical oscillations are visible.")
print("Near the limb (μ→0), supergranulation's horizontal flow dominates.")
print("The paper discovered BOTH by observing at different disk positions!")

## 요약 / Summary

| Part | 구현 내용 / Implementation | 논문 대응 / Paper Reference |
|---|---|---|
| 1 | Line-shifter 원리 — Doppler shift → 밝기 차이 변환 | Section I, Fig. 1 |
| 2 | Supergranulation 속도장 생성 및 시선 투영 | Section IV(a), Figs. 6-8 |
| 3 | A-C 함수로 패턴 크기 측정 (FWHM ~15,000 km) | Section II, Fig. 7 |
| 4 | 밝기–속도 상관의 고도별 부호 반전 | Section IV(b), Table 1, Fig. 9 |
| 5 | 5분 감쇠 진동 및 Doppler difference plate | Section IV(d), Figs. 14-15 |
| 6 | 기계적 에너지 수송 추정 (~2 W/cm²) | Section IV(b), Eq. 9 |
| 7 | A-C 피크 높이의 시간 의존성 — 감쇠율 측정 | Figs. 21-22 |
| 8 | Supergranulation + 5분 진동 통합 속도장 시각화 | Synthesis of all results |

### 다음 논문과의 연결 / Connection to Next Papers

- **#14 Ulrich (1970)**: 5분 진동을 태양 내부의 공명 음향파(p-mode)로 해석 → **helioseismology 시작**
  Interprets 5-minute oscillations as resonant acoustic waves (p-modes) inside the Sun → **helioseismology begins**
- **#15 Deubner (1975)**: Ulrich의 p-mode 분산 관계를 관측적으로 확인
  Observationally confirms Ulrich's p-mode dispersion relation